# BRUV Fish Counting — Starter Notebook

**Goal:** Count the maximum number of *Caranx caballus* (green jack) in a single frame
from underwater BRUV videos.

**Kaggle competition:** https://www.kaggle.com/competitions/marine-conservation-with-migra-mar

This notebook covers:
1. Loading the label CSVs (from Kaggle or local)
2. Downloading videos from R2
3. Extracting and visualizing frames
4. Locating labeled frames in the video
5. A simple baseline approach

---

## 0. Setup

In [ ]:
!pip install -q boto3 tqdm opencv-python-headless matplotlib pandas numpy

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Image as IPImage

## 1. Load label data

The label CSVs (species counts per frame) come from the
[Kaggle competition](https://www.kaggle.com/competitions/marine-conservation-with-migra-mar).
The cell below auto-downloads them using the Kaggle API so you don't have to
download and upload them manually.

### Kaggle API setup (one-time)

1. **Create a Kaggle account** (if you don't have one) at [kaggle.com](https://www.kaggle.com)
2. **Join the competition:** go to the [competition page](https://www.kaggle.com/competitions/marine-conservation-with-migra-mar) and click **Join Competition** (you must accept the rules before the API will let you download data)
3. **Get your API key:** go to [kaggle.com/settings](https://www.kaggle.com/settings) → scroll to **API** → under **Legacy API Credentials**, click **Create New Token** — this downloads a `kaggle.json` file
4. **Upload `kaggle.json`** to the Colab file panel (drag and drop into the left sidebar)
5. **Run the cell below** — it configures the key and downloads the competition CSVs automatically

> **Important:** Use **Legacy API Credentials**, not "API Tokens (Recommended)".
> The newer token format requires a different library (`kagglehub`) and is not
> compatible with the standard `kaggle` CLI used here.

In [ ]:
import zipfile, json, shutil, subprocess

COMPETITION = "marine-conservation-with-migra-mar"
CSV_DIR = Path("kaggle_data")

# === Auto-detect environment ===
if (Path("/kaggle/input") / COMPETITION).exists():
    # Running on Kaggle — data is pre-mounted
    CSV_DIR = Path("/kaggle/input") / COMPETITION
    print(f"Running on Kaggle — CSVs at {CSV_DIR}")

elif not (CSV_DIR / "CumulativeMaxN.csv").exists():
    # Download via Kaggle API
    print("Downloading competition data from Kaggle...")

    # Look for kaggle.json — always prefer a freshly uploaded file
    # (overwrite any stale cached copy in ~/.kaggle/)
    dest = Path.home() / ".kaggle/kaggle.json"
    search_paths = [
        Path("/content/kaggle.json"),   # Colab file panel upload
        Path("kaggle.json"),            # current directory
        Path("../kaggle.json"),         # repo root (if running from track subdir)
    ]
    for candidate in search_paths:
        if candidate.exists():
            dest.parent.mkdir(exist_ok=True)
            shutil.copy2(candidate, dest)
            dest.chmod(0o600)
            print(f"Configured Kaggle credentials from {candidate}")
            break

    if not dest.exists():
        raise FileNotFoundError(
            "kaggle.json not found.\n"
            "Upload it to the Colab file panel (drag and drop), or place it in your working directory.\n"
            "Get it from: kaggle.com/settings → API → Legacy API Credentials → Create New Token"
        )

    subprocess.check_call(["pip", "install", "-q", "kaggle"])
    result = subprocess.run(
        ["kaggle", "competitions", "download", "-c", COMPETITION, "-p", "kaggle_data/"],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        msg = result.stderr.strip() or result.stdout.strip()
        hint = ""
        if "401" in msg or "Unauthorized" in msg:
            hint = (
                "\n\nThis usually means:\n"
                "  1. You haven't joined the competition yet — go to\n"
                "     https://www.kaggle.com/competitions/marine-conservation-with-migra-mar\n"
                "     and click 'Join Competition'\n"
                "  2. Or your kaggle.json credentials are invalid — re-download from\n"
                "     kaggle.com/settings → API → Legacy API Credentials → Create New Token"
            )
        raise RuntimeError(f"Kaggle download failed:\n{msg}{hint}")

    print(result.stdout)

    # Unzip
    zip_path = CSV_DIR / f"{COMPETITION}.zip"
    if zip_path.exists():
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(CSV_DIR)
        print(f"Extracted to {CSV_DIR}")
else:
    print(f"CSVs already present in {CSV_DIR}")

# === Load CSVs ===
cumulative = pd.read_csv(CSV_DIR / "CumulativeMaxN.csv")
time_first = pd.read_csv(CSV_DIR / "TimeFirstSeen.csv")

print(f"CumulativeMaxN: {cumulative.shape}")
print(f"TimeFirstSeen:  {time_first.shape}")
display(cumulative.head(10))

In [ ]:
# === Species summary ===
print("Species in the data:")
species = cumulative.groupby(["Family", "Genus", "Species"]).agg(
    max_count=("Cumulative MaxN", "max"),
    observations=("Cumulative MaxN", "count"),
).reset_index()
display(species)

print("\nFirst appearance times:")
display(time_first)

## 2. Download videos from R2

Videos are hosted on Cloudflare R2 (~65 GB total). Start with just one
sub-video (~4 GB) — you can always download more later.

**Credentials:** Organizers will provide a `participant-download.env` file. Upload it
to the Colab file panel (drag and drop), then run the next cells. Alternatively, edit
the placeholder values directly.

In [ ]:
# === Get r2_download.py helper ===
HELPER_URL = "https://raw.githubusercontent.com/SALA-AI-LATAM/hackathon-participants/main/r2_download.py"

if not Path("r2_download.py").exists():
    print("Downloading r2_download.py...")
    import urllib.request
    urllib.request.urlretrieve(HELPER_URL, "r2_download.py")
    print("Done.")
else:
    print("r2_download.py already present.")

In [ ]:
import os
from pathlib import Path

# === Get r2_download.py helper module (if running on Colab/Kaggle) ===
HELPER_URL = "https://raw.githubusercontent.com/SALA-AI-LATAM/hackathon-participants/main/r2_download.py"

if not Path("r2_download.py").exists():
    print("Downloading r2_download.py...")
    import urllib.request
    urllib.request.urlretrieve(HELPER_URL, "r2_download.py")
    print("Done.")
else:
    print("r2_download.py already present.")

# === Set R2 credentials ===
# Search common locations for the .env file
ENV_NAME = "participant-download.env"
ENV_SEARCH_PATHS = [
    Path(ENV_NAME),               # current directory (notebook folder)
    Path("..") / ENV_NAME,        # parent directory (repo root)
    Path("/workspace") / ENV_NAME,  # RunPod workspace root
]

env_file = None
for p in ENV_SEARCH_PATHS:
    if p.exists():
        env_file = p
        break

if env_file is not None:
    for line in open(env_file):
        line = line.strip()
        if line and not line.startswith("#"):
            key, val = line.removeprefix("export ").split("=", 1)
            os.environ[key] = val.strip('"')
    print(f"Credentials loaded from {env_file}")
elif "R2_ENDPOINT" in os.environ:
    print("Using pre-set environment variables")
else:
    # Option B: Paste credentials directly (organizers will provide these)
    os.environ["R2_ENDPOINT"] = "https://YOUR_ACCOUNT_ID.r2.cloudflarestorage.com"
    os.environ["R2_ACCESS_KEY_ID"] = "YOUR_ACCESS_KEY"
    os.environ["R2_SECRET_ACCESS_KEY"] = "YOUR_SECRET_KEY"
    os.environ["R2_BUCKET"] = "sala-2026-hackathon-data"
    print("Using inline credentials (edit the values above if they say YOUR_...)")

In [ ]:
import r2_download as hd

client = hd.get_s3_client()
manifest = hd.load_manifest(
    bucket=os.environ["R2_BUCKET"], s3_client=client, cache_path="manifest.json"
)

# === Download just one sub-video to get started (~4 GB) ===
# LGH020002.MP4 is in vid2 (deployment 2), sub-video 02
stats = hd.download_dataset(
    manifest,
    dataset_name="bruv-videos",
    tags=["vid2-sub02"],  # LGH020002.MP4 — most action
)
print(f"Downloaded: {stats['downloaded']}, Skipped: {stats['skipped']}")

## 3. Extract and visualize frames

Use OpenCV to pull frames from the video. At 30fps, an 11-minute sub-video
has ~20,000 frames — start by sampling at 1 fps.

In [ ]:
# === Open a video and extract basic info ===
VIDEO_DIR = Path(hd._default_data_dir()) / "bruv-videos"
video_path = VIDEO_DIR / "LGH020002.MP4"

cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration_sec = total_frames / fps

print(f"Video: {video_path.name}")
print(f"Resolution: {width}x{height}")
print(f"FPS: {fps}")
print(f"Total frames: {total_frames:,}")
print(f"Duration: {duration_sec/60:.1f} minutes")
cap.release()

In [ ]:
def extract_frame(video_path, frame_number):
    """Extract a single frame from a video by frame number."""
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise ValueError(f"Could not read frame {frame_number}")
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def extract_frames_at_interval(video_path, interval_sec=1.0, max_frames=10):
    """Extract frames at regular time intervals."""
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    interval_frames = int(fps * interval_sec)
    frames = []
    frame_idx = 0
    while len(frames) < max_frames:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            break
        frames.append((frame_idx, cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        frame_idx += interval_frames
    cap.release()
    return frames


# === Show sample frames (1 per minute) ===
samples = extract_frames_at_interval(video_path, interval_sec=60, max_frames=12)

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
for ax, (idx, frame) in zip(axes.flat, samples):
    ax.imshow(frame)
    ax.set_title(f"Frame {idx:,} ({idx/fps/60:.1f} min)", fontsize=10)
    ax.axis("off")
for ax in axes.flat[len(samples):]:
    ax.axis("off")
plt.suptitle(f"{video_path.name} — 1 frame per minute", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Locate labeled frames

The CSV timestamps use the original (unsegmented) video timeline.
Each sub-video is ~11.783 minutes. Let's find and visualize the frames
where species were counted.

In [ ]:
SUB_VIDEO_DURATION_MIN = 11.783  # approximate duration of each sub-video

def global_time_to_local(time_mins, sub_video_name):
    """Convert global video time to local sub-video time.
    
    The sub-video index is encoded in the filename: LGH020002.MP4 = sub-video 2.
    Local time = global time - (sub_video_index - 1) * sub_video_duration.
    """
    # Extract sub-video index from filename (e.g., LGH020002 -> 02 -> index 2)
    idx = int(sub_video_name[3:5])  # e.g., '02' from 'LGH020002'
    local_time = time_mins - (idx - 1) * SUB_VIDEO_DURATION_MIN
    return local_time


# === Find labeled frames in LGH020002 ===
vid_labels = cumulative[cumulative["Filename"] == "LGH020002.MP4"].copy()
vid_labels["local_time_min"] = vid_labels.apply(
    lambda r: global_time_to_local(r["Time (mins)"], r["Filename"].replace(".MP4", "")),
    axis=1
)
vid_labels["local_frame"] = (vid_labels["local_time_min"] * 60 * fps).astype(int)

print(f"Labeled events in LGH020002.MP4: {len(vid_labels)}")
display(vid_labels[["Species", "Frame", "local_time_min", "local_frame", "Cumulative MaxN"]])

In [ ]:
# === Visualize the labeled frames ===
# Use local_frame (converted from global time) rather than the raw Frame column,
# which may use the original unsegmented video's frame numbering
n_show = min(8, len(vid_labels))
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for ax, (_, row) in zip(axes.flat, vid_labels.head(n_show).iterrows()):
    frame = extract_frame(video_path, row["local_frame"])
    ax.imshow(frame)
    species_short = row["Species"]
    count = row["Cumulative MaxN"]
    ax.set_title(f"{species_short} (n={count})\nFrame {row['local_frame']:,}", fontsize=10)
    ax.axis("off")

for ax in axes.flat[n_show:]:
    ax.axis("off")
plt.suptitle("Labeled frames from LGH020002.MP4", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Baseline: Background subtraction

Since the BRUV is stationary, anything moving is likely a fish. OpenCV's
background subtractor can isolate motion. This won't count species, but
it's a useful first step to detect when fish are present.

In [ ]:
from tqdm import tqdm

def compute_motion_signal(video_path, sample_fps=2, max_seconds=None):
    """Compute a motion intensity signal over time using background subtraction.
    
    Returns arrays of (times_sec, motion_intensity) where motion_intensity
    is the fraction of pixels classified as foreground.
    """
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    interval = int(fps / sample_fps)
    
    if max_seconds:
        total_frames = min(total_frames, int(max_seconds * fps))
    
    bg_sub = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50)
    
    times = []
    motion = []
    
    n_samples = total_frames // interval
    for i in tqdm(range(0, total_frames, interval), total=n_samples, desc="Motion signal"):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret:
            break
        
        # === Apply background subtraction ===
        mask = bg_sub.apply(frame)
        motion_frac = np.count_nonzero(mask) / mask.size
        
        times.append(i / fps)
        motion.append(motion_frac)
    
    cap.release()
    return np.array(times), np.array(motion)


# === Compute motion signal for the first 5 minutes ===
times, motion = compute_motion_signal(video_path, sample_fps=2, max_seconds=300)

plt.figure(figsize=(14, 4))
plt.plot(times / 60, motion, linewidth=0.8)
plt.xlabel("Time (minutes)")
plt.ylabel("Motion intensity (foreground fraction)")
plt.title(f"{video_path.name} — Background subtraction motion signal")

# === Overlay labeled event times ===
for _, row in vid_labels.iterrows():
    local_sec = row["local_time_min"] * 60
    if local_sec <= 300:
        plt.axvline(local_sec / 60, color="red", alpha=0.5, linestyle="--", linewidth=1)

plt.legend(["Motion signal", "Labeled events"])
plt.tight_layout()
plt.show()

## 6. Next steps

You now have:
- Label CSVs loaded and explored
- Video frames extracted and visualized
- A simple motion baseline

From here, some directions:

- **Object detection:** Run YOLOv8 (`pip install ultralytics`) on extracted frames.
  Pre-trained COCO weights detect generic objects — fish may show up as "bird" or other categories.
  Fine-tuning on a few annotated frames will dramatically improve results.

- **Crowd counting:** With 251 fish in a single frame, this is a dense counting problem.
  Look at CSRNet or density map approaches.

- **Tracking:** Use ByteTrack or SORT to track individual fish across frames.
  This helps avoid double-counting and can give you MaxN more reliably.

- **VLM zero-shot:** Try asking GPT-4V or Gemini to count fish in a frame.
  Surprisingly effective for moderate counts, less so for 200+ fish.

**Remember:** The Kaggle competition values contributions at any step of the
pipeline, not just the final count. A clever video processing approach or a
good fish detector is valuable even without a complete MaxN solution.

**Submit your results:** https://www.kaggle.com/competitions/marine-conservation-with-migra-mar